## MobileNet Training and Fine-Tuning on the Cleaned FANE Image Dataset

In [7]:
import os
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint

## 2. Load cleaned dataset
Prepare the cleaned dataset directories and load image datasets for training, validation, and testing.

In [24]:
project_path = os.path.dirname(os.getcwd())
cleaned_dir = os.path.join(project_path, "Data", "Computer-Vision", "fane_cleaned")
train_dir = os.path.join(cleaned_dir, "train")
val_dir   = os.path.join(cleaned_dir, "val")
test_dir  = os.path.join(cleaned_dir, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 7844 files belonging to 9 classes.
Found 1683 files belonging to 9 classes.
Found 1686 files belonging to 9 classes.


In [25]:
class_names = train_ds.class_names
print("Class names:", class_names)

Class names: ['angry', 'confused', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'shy', 'surprise']


## 3. Prepare dataset pipeline
Optimize the dataset pipeline with prefetching for better training performance.

In [9]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.prefetch(buffer_size=AUTOTUNE)

In [10]:
for images, labels in train_ds.take(1):
    print("Images batch shape:", images.shape)
    print("Labels batch shape:", labels.shape)
    print("First 10 labels:", labels[:10].numpy())

Images batch shape: (32, 224, 224, 3)
Labels batch shape: (32,)
First 10 labels: [8 8 6 6 8 6 0 0 5 8]


## 4. Build transfer learning model
Define a MobileNetV2-based model with data augmentation and a custom classification head.

In [11]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
])

In [12]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

In [13]:
NUM_CLASSES = 9
inputs = layers.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

## 5. Define training callbacks
Create callbacks for early stopping, learning-rate reduction, and best-model checkpointing.

In [ ]:
initial_epochs = 10

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-8
)
checkpoint = ModelCheckpoint(
    "best_frozen_model.keras", 
    monitor='val_accuracy', 
    save_best_only=True
)

In [ ]:
# # --- Compute simple class weights from the training folders ---
# # This mitigates class imbalance by up-weighting rare classes
# class_counts = {}
# total = 0
# classes_sorted = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
# for i, cls in enumerate(classes_sorted):
#     cnt = len([f for f in os.listdir(os.path.join(train_dir, cls)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
#     class_counts[i] = cnt
#     total += cnt
# # weight = total / (num_classes * count)
# class_weights = {i: (total / (len(class_counts) * cnt)) for i, cnt in class_counts.items()}
# print('Computed class weights:', class_weights)

## 6. Train model
Train the frozen MobileNetV2 feature extractor before fine-tuning.

In [18]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [19]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=initial_epochs,
    callbacks=[early_stop, reduce_lr, checkpoint],
)

Epoch 1/10


246/246 ━━━━━━━━━━━━━━━━━━━━ 337s 1s/step - accuracy: 0.3001 - loss: 1.9504 - val_accuracy: 0.3256 - val_loss: 1.8397 - learning_rate: 1.0000e-06
Epoch 2/10
246/246 ━━━━━━━━━━━━━━━━━━━━ 252s 1s/step - accuracy: 0.3093 - loss: 1.9476 - val_accuracy: 0.3226 - val_loss: 1.8391 - learning_rate: 1.0000e-06
Epoch 3/10
246/246 ━━━━━━━━━━━━━━━━━━━━ 292s 1s/step - accuracy: 0.3041 - loss: 1.9508 - val_accuracy: 0.3220 - val_loss: 1.8387 - learning_rate: 1.0000e-06
Epoch 4/10
246/246 ━━━━━━━━━━━━━━━━━━━━ 290s 1s/step - accuracy: 0.3029 - loss: 1.9495 - val_accuracy: 0.3232 - val_loss: 1.8384 - learning_rate: 1.0000e-06
Epoch 5/10
246/246 ━━━━━━━━━━━━━━━━━━━━ 254s 1s/step - accuracy: 0.3107 - loss: 1.9284 - val_accuracy: 0.3238 - val_loss: 1.8381 - learning_rate: 1.0000e-06
Epoch 6/10
246/246 ━━━━━━━━━━━━━━━━━━━━ 232s 944ms/step - accuracy: 0.3041 - loss: 1.9479 - val_accuracy: 0.3238 - val_loss: 1.8377 - learning_rate: 1.0000e-06
Epoch 7/10
246/246 ━━━━━━━━━━━━━━━━━━━━ 246s 1s/step - accuracy: 0

In [ ]:
model.summary()

## 7. Fine-tune top layers
Unfreeze the top layers of MobileNetV2 and continue training with a lower learning rate.

In [20]:
# --- Fine-tuning: unfreeze top layers of base_model and continue training ---
# Unfreeze the top N layers for fine-tuning (keep most layers frozen)
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with a lower learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

fine_epochs = 10
total_epochs = initial_epochs + fine_epochs
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] if hasattr(history, 'epoch') else 0,
    callbacks=[early_stop, reduce_lr, checkpoint]
)

Epoch 10/20
246/246 ━━━━━━━━━━━━━━━━━━━━ 303s 1s/step - accuracy: 0.2478 - loss: 2.1626 - val_accuracy: 0.3185 - val_loss: 1.9892 - learning_rate: 1.0000e-05
Epoch 11/20
246/246 ━━━━━━━━━━━━━━━━━━━━ 274s 1s/step - accuracy: 0.3185 - loss: 1.9147 - val_accuracy: 0.3452 - val_loss: 1.9224 - learning_rate: 1.0000e-05
Epoch 12/20
246/246 ━━━━━━━━━━━━━━━━━━━━ 288s 1s/step - accuracy: 0.3506 - loss: 1.7996 - val_accuracy: 0.3613 - val_loss: 1.8300 - learning_rate: 1.0000e-05
Epoch 13/20
246/246 ━━━━━━━━━━━━━━━━━━━━ 287s 1s/step - accuracy: 0.3840 - loss: 1.7259 - val_accuracy: 0.3791 - val_loss: 1.7647 - learning_rate: 1.0000e-05
Epoch 14/20
246/246 ━━━━━━━━━━━━━━━━━━━━ 285s 1s/step - accuracy: 0.3948 - loss: 1.6875 - val_accuracy: 0.3969 - val_loss: 1.7203 - learning_rate: 1.0000e-05
Epoch 15/20
246/246 ━━━━━━━━━━━━━━━━━━━━ 293s 1s/step - accuracy: 0.4245 - loss: 1.6310 - val_accuracy: 0.4100 - val_loss: 1.6839 - learning_rate: 1.0000e-05
Epoch 16/20
246/246 ━━━━━━━━━━━━━━━━━━━━ 277s 1s/ste

## 8. Evaluate model
Load the best checkpoint and compute performance metrics on the test set.

In [22]:
# --- Evaluation: load best checkpoint and produce detailed metrics ---
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Load best saved model
model = tf.keras.models.load_model('best_frozen_model.keras')
test_loss, test_acc = model.evaluate(test_ds)
print('Test loss:', test_loss, 'Test accuracy:', test_acc)

# Build y_true and y_pred lists for reporting
y_true = []
y_pred = []
for images, labels in test_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(preds, axis=1).tolist())

print(classification_report(y_true, y_pred))

53/53 ━━━━━━━━━━━━━━━━━━━━ 41s 712ms/step - accuracy: 0.4828 - loss: 1.5222
Test loss: 1.5221846103668213 Test accuracy: 0.4827995300292969
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 791ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 816ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 730ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 722ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 716ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 709ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 697ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 702ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 738ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 728ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 699ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 719ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 703ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 718ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 706ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 691ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 818ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 781ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 773ms/step
1/1 ━

In [27]:
report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
)

print(report)

              precision    recall  f1-score   support

       angry     0.4784    0.4512    0.4644       246
    confused     0.4077    0.3029    0.3475       175
     disgust     0.4162    0.3913    0.4034       184
        fear     0.4756    0.2690    0.3436       145
       happy     0.5319    0.7485    0.6219       167
     neutral     0.5344    0.8398    0.6532       231
         sad     0.3750    0.1548    0.2192       155
         shy     0.4706    0.2857    0.3556       140
    surprise     0.4845    0.6420    0.5522       243

    accuracy                         0.4828      1686
   macro avg     0.4638    0.4539    0.4401      1686
weighted avg     0.4677    0.4828    0.4578      1686

